# 2025-10-20: Edit cell labels, add CD8+T Central Memory, and Plasma Cell malignancy Correction
### By [Aishwarya Chander](aishwarya.chander@alleninstitute.org), High Resolution Translational Immunology, Allen Institute for Immunology
**Main aim**:Rebuild the full L3→L2→L1.5→L1 cell type hierarchy incorporating plasma cell malignancy status (malignant vs non-malignant from David Glass's isotype analysis), T cell label corrections (adding CD8+ central memory), and tissue-resident CD8 T cells as a distinct L2 category. Generates abbreviated plot labels (aifi_plot_l2) for downstream visualization.

In [1]:
import pandas as pd

## 1. Get labels from Plasma Cell clean up and outlier clean up

In [2]:
plasma_subset = (
    pd.read_csv("../../../data/rna/metadata/david_glass-mm_asc_identity_dat.csv")
    .rename({"cell_uuid": "barcodes", "malignant": "aifi_celltype_l3"}, axis=1)
    .set_index("barcodes")
    .drop(["subisotype", "isotype"], axis=1)
)

plasma_subset['aifi_celltype_l3'] = plasma_subset['aifi_celltype_l3'].map({True: 'malignant', False: 'non-malignant'})
plasma_subset["junk_cells"] = 0

In [3]:
all_labels = pd.read_parquet(
    "../../../data/rna/bmmc-labels/nonplasma_cleaned_labels.parquet"
)
all_labels.rename(columns={"aifi_celltype_l3_knn": "aifi_celltype_l3"}, inplace=True)

In [4]:
all_labels['aifi_celltype_l3'].value_counts()

t_cd4_naive             180017
mono_cd14               115945
t_cd4_central_memory     78978
b_naive                  77902
t_cd8_effector_2         58371
                         ...  
b_transitional_isg         866
dc_asdc                    799
prog_megakaryocyte         469
prog_hspc_cycling          456
dc_cdc2-isg.pos            288
Name: aifi_celltype_l3, Length: 62, dtype: int64

In [5]:
all_labels = pd.concat([all_labels, plasma_subset])
all_labels["junk_cells"] = all_labels["junk_cells"].astype(bool)

## 2. Add L3 Cell Types

### 2.1. Clean up L3 to match [AIFI L3 PBMC Reference](https://docs.google.com/spreadsheets/d/1A8H4y2LzDqhE7SjVUFwDGHD3N6Ns9lWcHz9Rm-y3xEc/edit?gid=1103786458#gid=1103786458) as much as possible

In [6]:
messy_to_l3 = {
    "b_memory_cd95": "b_memory-cd95",
    "b_memory_core": "b_memory-core",
    "b_naive": "b_naive-core",
    "b_naive_isg": "b_naive-isg.pos",
    "b_precursor_hcr": "b_precursor_heavy_chain",
    "b_precursor_isg": "b_precursor-isg.pos",
    "b_precursor_lcr": "b_precursor_light_chain",
    "b_precursor_proliferating": "b_precursor_proliferating",
    "b_transitional": "b_transitional-core",
    "b_transitional_isg": "b_transitional-isg.pos",
    "dc_asdc": "dc_asdc",
    "dc_cdc1": "dc_cdc1",
    "dc_cdc2": "dc_cdc2-core",
    "dc_cdc2-isg.pos": "dc_cdc2-isg.pos",
    "dc_pdc": "dc_pdc",
    "non-malignant": "plasma_non-malignant",
    "mono_cd14": "mono_cd14-core",
    "mono_cd14_isg": "mono_cd14-isg.pos",
    "mono_cd16": "mono_cd16-core",
    "mono_intermediate": "mono_intermediate",
    "mono_precursor": "mono_precursor-core",
    "mono_precursor_proliferating": "mono_precursor_proliferating",
    "nk_adaptive": "nk_adaptive",
    "nk_cd56_bright": "nk_cd56_bright",
    "nk_cd56_dim-gzmk_neg": "nk_cd56.dim-gzmk.neg",
    "nk_cd56_dim-gzmk_pos": "nk_cd56.dim-gzmk.pos",
    "nk_cd56_dim-isg_pos": "nk_cd56.dim-isg.pos",
    "nk_effector": "nk_effector",
    "nk_t_proliferating_nk_like": "nk_proliferating",
    "nk_t_proliferating_t_like": "t_proliferating",
    "nk_tissue_resident": "nk_tissue_resident",
    "malignant": "plasma_malignant",
    "prog_b_early": "prog_b_precursor",
    "prog_b_late": "prog_b_mature",
    "prog_b_proliferating": "prog_b_proliferating",
    "prog_baeoma": "prog_ba-eo-ma",
    "prog_clp": "prog_clp",
    "prog_cmp": "prog_cmp_granulocyte",
    "prog_cmp_granulocyte": "prog_cmp-core",
    "prog_dc_cdc": "prog_dc_cdc",
    "prog_dc_pdc": "prog_dc_pdc",
    "prog_ery_cycling": "prog_ery_proliferating",
    "prog_ery_mature": "prog_ery_mature",
    "prog_ery_pre": "prog_ery_precursor",
    "prog_hspc_cycling": "prog_hspc_proliferating",
    "prog_hspc_multipotential": "prog_hspc_multipotential",
    "prog_hspc_stem": "prog_hspc_stem",
    "prog_lmpp": "prog_lmpp",
    "prog_megakaryocyte": "prog_megakaryocyte",
    "prog_mep": "prog_mep",
    "t_cd4_central_memory": "t_cd4_memory_central",
    "t_cd4_effector_1": "t_cd4_memory_effector_1",
    "t_cd4_effector_2": "t_cd4_memory_effector_2",
    "t_cd4_isg": "t_cd4_naive-isg.pos",
    "t_cd4_memory": "t_cd4_memory-core",
    "t_cd4_naive": "t_cd4_naive-core",
    "t_cd4_regs": "t_cd4_regs",
    "t_cd8_effector_1": "t_cd8_memory_effector_1",
    "t_cd8_effector_2": "t_cd8_memory_effector_2",
    "t_cd8_memory_tissue_resident": "t_cd8_memory_tissue_resident",
    "t_cd8_naive": "t_cd8_naive-core",
    "t_dn": "t_dn",
    "t_gd": "t_gd",
    "t_mait": "t_mait",
}

all_labels["aifi_celltype_l3"] = (
    all_labels["aifi_celltype_l3"]
    .map(messy_to_l3)
    .astype("category")
    .cat.remove_unused_categories()
)

### Add in the edited T Cell Subset

In [7]:
t_edits = pd.read_parquet("../../../data/rna/bmmc-labels/t_cells_corrected.parquet")
all_labels["aifi_celltype_l3"] = all_labels["aifi_celltype_l3"].astype(object)
all_labels.loc[all_labels.index.intersection(t_edits.index), "aifi_celltype_l3"] = (
    t_edits.loc[all_labels.index.intersection(t_edits.index), "aifi_celltype_l3"]
)
all_labels["aifi_celltype_l3"] = all_labels["aifi_celltype_l3"].astype("category")

### 2.2. Add L3 Labels

In [8]:
l3_to_label3 = {
    "b_memory-cd95": "CD95 memory B cell",
    "b_memory-core": "Core memory B cell",
    "b_naive-core": "Core naive B cell",
    "b_naive-isg.pos": "ISG+ naive B cell",
    "b_precursor_heavy_chain": "Heavy chain recombinant precursor B cell",
    "b_precursor_light_chain": "Light chain recombinant precursor B cell",
    "b_precursor_proliferating": "Proliferating precursor B cell",
    "b_precursor-isg.pos": "ISG+ precursor B cell",
    "b_transitional-core": "Transitional B cell",
    "b_transitional-isg.pos": "ISG+ Transitional B cell",
    "dc_asdc": "ASDC",
    "dc_cdc1": "cDC1",
    "dc_cdc2-core": "cDC2",
    "dc_cdc2-isg.pos": "ISG+ cDC2",
    "dc_pdc": "pDC",
    "mono_cd14-core": "Core CD14 monocyte",
    "mono_cd14-isg.pos": "ISG+ CD14 monocyte",
    "mono_cd16-core": "Core CD16 monocyte",
    "mono_intermediate": "Intermediate monocyte",
    "mono_precursor_proliferating": "Proliferating precursor monocyte",
    "mono_precursor-core": "Precursor monocyte",
    "nk_adaptive": "Adaptive NK cell",
    "nk_cd56_bright": "CD56bright NK cell",
    "nk_cd56.dim-gzmk.neg": "GZMK- CD56dim NK cell",
    "nk_cd56.dim-gzmk.pos": "GZMK+ CD56dim NK cell",
    "nk_cd56.dim-isg.pos": "ISG+ CD56dim NK cell",
    "nk_effector": "Effector NK cell",
    "nk_proliferating": "Proliferating NK cell ",
    "nk_tissue_resident": "Tissue resident NK cell",
    "plasma_non-malignant": "Plasma (Non-malignant)",
    "plasma_malignant": "Plasma (Malignant)",
    "prog_b_mature": "Mature B cell progenitor",
    "prog_b_precursor": "Precursor B cell progenitor",
    "prog_b_proliferating": "B cell progenitor",
    "prog_ba-eo-ma": "BaEoMaP cell",
    "prog_clp": "Common lymphoid progenitor",
    "prog_cmp_granulocyte": "Granulocyte progenitor",
    "prog_cmp-core": "Common myeloid progenitor",
    "prog_dc_cdc": "cDC commited DC progenitor",
    "prog_dc_pdc": "pDC committed DC progenitor",
    "prog_ery_mature": "Mature erythroid progenitor",
    "prog_ery_precursor": "Precursor erythroid progenitor",
    "prog_ery_proliferating": "Proliferating erythroid progenitor",
    "prog_hspc_multipotential": "Multipotential HSPC",
    "prog_hspc_proliferating": "Proliferating HSPC",
    "prog_hspc_stem": "Hematopoetic stem and progenitor",
    "prog_lmpp": "Lymphoid-primed multipotent progenitor",
    "prog_megakaryocyte": "Megakaryocyte progenitor cell",
    "prog_mep": "Megakaryocyte-erythroid progenitor cell",
    "t_cd4_memory_central": "CM CD4 T cell",
    "t_cd4_memory_effector_1": "CD4 Effector Memory 1 T cell",
    "t_cd4_memory_effector_2": "CD4 Effector Memory 2 T cell",
    "t_cd4_memory-core": "CD4 Memory T cell",
    "t_cd4_naive-core": "Core naive CD4 T cell",
    "t_cd4_naive-isg.pos": "ISG+ naive CD4 T cell",
    "t_cd4_regs": "CD4 Treg",
    "t_cd8_memory_central": "CM CD8 T cell",
    "t_cd8_memory_effector_1": "CD8 Effector Memory 1 T cell",
    "t_cd8_memory_effector_2": "CD8 Effector Memory 2 T cell",
    "t_cd8_memory_tissue_resident": "Tissue resident CD8 T cell",
    "t_cd8_naive-core": "Core naive CD8 T cell ",
    "t_dn": "DN T cell",
    "t_gd": "gdT",
    "t_mait": "MAIT",
    "t_proliferating": "Proliferating T cell",
}

all_labels["aifi_label_l3"] = (
    all_labels["aifi_celltype_l3"]
    .map(l3_to_label3)
    .astype("category")
    .cat.remove_unused_categories()
)

## 3. Add L2 Cell Types

### 3.1. Map L3 cell types back to L2 cell types

In [9]:
l3_to_l2 = {
    "b_memory-cd95": "b_memory",
    "b_memory-core": "b_memory",
    "b_naive-core": "b_naive",
    "b_naive-isg.pos": "b_naive",
    "b_precursor_heavy_chain": "b_precursor",
    "b_precursor_light_chain": "b_precursor",
    "b_precursor_proliferating": "b_precursor",
    "b_precursor-isg.pos": "b_precursor",
    "b_transitional-core": "b_transitional",
    "b_transitional-isg.pos": "b_transitional",
    "dc_asdc": "dc_asdc",
    "dc_cdc1": "dc_cdc1",
    "dc_cdc2-core": "dc_cdc2",
    "dc_cdc2-isg.pos": "dc_cdc2",
    "dc_pdc": "dc_pdc",
    "mono_cd14-core": "mono_cd14",
    "mono_cd14-isg.pos": "mono_cd14",
    "mono_cd16-core": "mono_cd16",
    "mono_intermediate": "mono_intermediate",
    "mono_precursor_proliferating": "mono_precursor",
    "mono_precursor-core": "mono_precursor",
    "nk_adaptive": "nk_cd56_dim",
    "nk_cd56_bright": "nk_cd56_bright",
    "nk_cd56.dim-gzmk.neg": "nk_cd56_dim",
    "nk_cd56.dim-gzmk.pos": "nk_cd56_dim",
    "nk_cd56.dim-isg.pos": "nk_cd56_dim",
    "nk_effector": "nk_cd56_dim",
    "nk_proliferating": "nk_proliferating",
    "nk_tissue_resident": "nk_tissue_resident",
    "plasma_non-malignant": "plasma_non-malignant",
    "plasma_malignant": "plasma_malignant",
    "prog_b_mature": "prog_b",
    "prog_b_precursor": "prog_b",
    "prog_b_proliferating": "prog_b",
    "prog_ba-eo-ma": "prog_ba-eo-ma",
    "prog_clp": "prog_clp",
    "prog_cmp_granulocyte": "prog_cmp",
    "prog_cmp-core": "prog_cmp",
    "prog_dc_cdc": "prog_dc",
    "prog_dc_pdc": "prog_dc",
    "prog_ery_mature": "prog_mature_ery",
    "prog_ery_precursor": "prog_ery",
    "prog_ery_proliferating": "prog_mature_ery",
    "prog_hspc_multipotential": "prog_hspc",
    "prog_hspc_proliferating": "prog_hspc",
    "prog_hspc_stem": "prog_hspc",
    "prog_lmpp": "prog_lmpp",
    "prog_megakaryocyte": "prog_mk",
    "prog_mep": "prog_mep",
    "t_cd4_memory_central": "t_cd4_memory",
    "t_cd4_memory_effector_1": "t_cd4_memory",
    "t_cd4_memory_effector_2": "t_cd4_memory",
    "t_cd4_memory-core": "t_cd4_memory",
    "t_cd4_naive-core": "t_cd4_naive",
    "t_cd4_naive-isg.pos": "t_cd4_naive",
    "t_cd4_regs": "t_cd4_regs",
    "t_cd8_memory_effector_1": "t_cd8_memory",
    "t_cd8_memory_effector_2": "t_cd8_memory",
    "t_cd8_memory_tissue_resident": "t_cd8_memory_tissue_resident",
    "t_cd8_naive-core": "t_cd8_naive",
    "t_dn": "t_dn",
    "t_gd": "t_gd",
    "t_mait": "t_mait",
    "t_proliferating": "t_proliferating",
}

all_labels["aifi_celltype_l2"] = (
    all_labels["aifi_celltype_l3"]
    .map(l3_to_l2)
    .astype("category")
    .cat.remove_unused_categories()
)

### 3.2. Map L2 cell types to L2 labels

In [10]:
l2_to_label2 = {
    "b_memory": "Memory B cell",
    "b_naive": "Naive B cell",
    "b_precursor": "Precursor B cell",
    "b_transitional": "Transitional B cell",
    "dc_asdc": "ASDC",
    "dc_cdc1": "cDC1",
    "dc_cdc2": "cDC2",
    "dc_pdc": "pDC",
    "mono_cd14": "Core CD14 monocyte",
    "mono_cd16": "Core CD16 monocyte",
    "mono_intermediate": "Intermediate monocyte",
    "mono_precursor": "Precursor monocyte",
    "nk_cd56_dim": "CD56dim NK cell",
    "nk_cd56_bright": "CD56bright NK cell",
    "nk_proliferating": "Proliferating NK cell ",
    "nk_tissue_resident": "Tissue resident NK cell",
    "plasma_non-malignant": "Plasma (Non-malignant)",
    "plasma_malignant": "Plasma (Malignant)",
    "prog_b": "B cell progenitor",
    "prog_ba-eo-ma": "BaEoMaP cell",
    "prog_clp": "Common lymphoid progenitor cell",
    "prog_cmp": "Common myeloid progenitor cell",
    "prog_dc": "Dendritic cell progenitor",
    "prog_mature_ery": "Mature erythroid progenitor cell",
    "prog_ery": "Precursor erythroid progenitor cell",
    "prog_hspc": "Hematopoetic stem and progenitor cell",
    "prog_lmpp": "Lymphoid-primed multipotent progenitor cell",
    "prog_mk": "Megakaryocyte progenitor cell",
    "prog_mep": "Megakaryocyte-erythroid progenitor cell",
    "t_cd8_memory_tissue_resident":  "Tissue resident T cell",
    "t_cd4_memory": "Memory CD4 T cell",
    "t_cd4_naive": "Naive CD4 T cell",
    "t_cd4_regs": "CD4 Treg",
    "t_cd8_memory": "CD8 Memory T cell",
    "t_cd8_naive": "Naive CD8 T cell",
    "t_dn": "DN T cell",
    "t_gd": "gdT",
    "t_mait": "MAIT",
    "t_proliferating": "Proliferating T cell",
}

all_labels["aifi_label_l2"] = (
    all_labels["aifi_celltype_l2"]
    .map(l2_to_label2)
    .astype("category")
    .cat.remove_unused_categories()
)

## 4. Add L1.5 Cell Types

### 4.1. Map L2 cell types back to L1.5 cell types

In [11]:
l2_to_l15 = {
    "b_memory": "b_memory",
    "b_naive": "b_naive",
    "b_precursor": "b_precursor",
    "b_transitional": "b_transitional",
    "dc_asdc": "dc_asdc",
    "dc_cdc1": "dc_cdc",
    "dc_cdc2": "dc_cdc",
    "dc_pdc": "dc_pdc",
    "mono_cd14": "mono_cd14",
    "mono_cd16": "mono_cd16",
    "mono_intermediate": "mono_intermediate",
    "mono_precursor": "mono_precursor",
    "nk_cd56_dim": "nk_cd56",
    "nk_cd56_bright": "nk_cd56",
    "nk_proliferating": "nk_proliferating",
    "nk_tissue_resident": "nk_tissue_resident",
    "plasma_non-malignant": "plasma",
    "plasma_malignant": "plasma",
    "prog_b": "prog_b",
    "prog_ba-eo-ma": "prog_cmp",
    "prog_clp": "prog_clp",
    "prog_cmp": "prog_cmp",
    "prog_dc": "prog_cmp",
    "prog_mature_ery": "prog_ery",
    "prog_ery": "prog_ery",
    "prog_hspc": "prog_hspc",
    "prog_lmpp": "prog_lmpp",
    "prog_mk": "prog_mk",
    "prog_mep": "prog_mk",
    "t_cd4_memory": "t_cd4",
    "t_cd4_naive": "t_cd4",
    "t_cd4_regs": "t_cd4",
    "t_cd8_memory": "t_cd8",
    "t_cd8_memory_tissue_resident": "t_cd8",
    "t_cd8_naive": "t_cd8",
    "t_dn": "t_other",
    "t_gd": "t_other",
    "t_mait": "t_other",
    "t_proliferating": "t_proliferating",
}

all_labels["aifi_celltype_l1.5"] = (
    all_labels["aifi_celltype_l2"]
    .map(l2_to_l15)
    .astype("category")
    .cat.remove_unused_categories()
)

### 4.2. Map L1.5 cell types to L1.5 labels

In [12]:
l15_to_label15 = {
    'b_memory': 'Memory B cell',
    'b_naive': 'Naive B cell',
    'b_precursor': 'Precursor B cell',
    'b_transitional': 'Transitional B cell',
    'dc_asdc': 'ASDC',
    'dc_cdc': 'cDC',
    'dc_pdc': 'pDC',
    'mono_cd14': 'Core CD14 monocyte',
    'mono_cd16': 'Core CD16 monocyte',
    'mono_intermediate': 'Intermediate monocyte',
    'mono_precursor': 'Precursor monocyte',
    'nk_cd56': 'CD56 NK Cell',
    'nk_proliferating': 'Proliferating NK cell ',
    'nk_tissue_resident': 'Tissue resident NK cell',
    'plasma': 'Plasma cell',
    'prog_b': 'B cell progenitor',
    'prog_cmp': 'Common myeloid progenitor cell',
    'prog_clp': 'Common lymphoid progenitor cell',
    'prog_ery': 'Erythroid Progenitor',
    'prog_hspc': 'Hematopoetic stem and progenitor cell',
    'prog_lmpp': 'Lymphoid-primed multipotent progenitor cell',
    'prog_mk': 'Megakaryocyte progenitor cell',
    't_cd4': 'CD4 T cell',
    't_cd8': 'CD8 T cell',
    't_other': 'Other T cell',
    't_proliferating': 'Proliferating T cell'
}

all_labels['aifi_label_l1.5'] = all_labels['aifi_celltype_l1.5'].map(
    l15_to_label15).astype('category').cat.remove_unused_categories()

## 5. Add L1 Celltypes

### 5.1. Map L1.5 cell types to L1 cell types

In [13]:
l15_to_l1 = {
    "b_memory": "b_cell",
    "b_naive": "b_cell",
    "b_precursor": "b_cell",
    "b_transitional": "b_cell",
    "dc_asdc": "dc",
    "dc_cdc": "dc",
    "dc_pdc": "dc",
    "mono_cd14": "monocyte",
    "mono_cd16": "monocyte",
    "mono_intermediate": "monocyte",
    "mono_precursor": "monocyte",
    "nk_cd56": "nk_cell",
    "nk_proliferating": "nk_cell",
    "nk_tissue_resident": "nk_cell",
    "plasma": "plasma",
    "prog_b": "progenitor",
    "prog_cmp": "progenitor",
    "prog_clp": "progenitor",
    "prog_ery": "progenitor",
    "prog_hspc": "progenitor",
    "prog_lmpp": "progenitor",
    "prog_mk": "progenitor",
    "t_cd4": "t_cell",
    "t_cd8": "t_cell",
    "t_other": "t_cell",
    "t_proliferating": "t_cell",
}

all_labels["aifi_celltype_l1"] = (
    all_labels["aifi_celltype_l1.5"]
    .map(l15_to_l1)
    .astype("category")
    .cat.remove_unused_categories()
)

### 5.2. Map L1 cell types to L1 labels

In [14]:
l1_to_label1 = {
    "b_cell": "B cell",
    "dc": "DC",
    "monocyte": "Monocyte",
    "nk_cell": "NK cell",
    "plasma": "Plasma cell",
    "progenitor": "Progenitor cell",
    "t_cell": "T Cell",
}

all_labels["aifi_label_l1"] = (
    all_labels["aifi_celltype_l1"]
    .map(l1_to_label1)
    .astype("category")
    .cat.remove_unused_categories()
)

## 6. Create Plotting/Analysis labels

In [15]:
l2_to_plot2 = {
    # T cell
    "gdT": "gdT",
    "Memory CD4 T cell": "CD4 T Memory",
    "CD8 Memory T cell": "CD8 T Memory",
    "Naive CD4 T cell": "CD4 T Naive",
    "CD4 Treg": "Treg",
    "MAIT": "MAIT",
    "Naive CD8 T cell": "CD8 T Naive",
    "DN T cell": "DN T",
    "Tissue resident T cell": "Tissue Res T",
    "Proliferating T cell": "Prolif T",
    
    # Monocyte
    "Core CD16 monocyte": "CD16 Mono",
    "Precursor monocyte": "Pre Mono",
    "Core CD14 monocyte": "CD14 Mono",
    "Intermediate monocyte": "Int Mono",
    
    # Progenitor cell
    "B cell progenitor": "Prog B",
    "Common lymphoid progenitor cell": "CLP",
    "Megakaryocyte-erythroid progenitor cell": "MEP",
    "BaEoMaP cell": "BaEoMaP",
    "Mature erythroid progenitor cell": "Prog Ery",
    "Dendritic cell progenitor": "Prog DC",
    "Common myeloid progenitor cell": "CMP",
    "Lymphoid-primed multipotent progenitor cell": "LMPP",
    "Precursor erythroid progenitor cell": "Pre Prog Ery",
    "Hematopoetic stem and progenitor cell": "HSPC",
    "Megakaryocyte progenitor cell": "Prog MK",
    
    # NK cell
    "CD56dim NK cell": "CD56dim NK",
    "CD56bright NK cell": "CD56br NK",
    "Tissue resident NK cell": "Tissue Res NK",
    "Proliferating NK cell ": "Prolif NK",
    
    # Dendritic cell
    "cDC2": "cDC2",
    "pDC": "pDC",
    "cDC1": "cDC1",
    "ASDC": "ASDC",
    
    # B cell
    "Precursor B cell": "Pre B",
    "Transitional B cell": "Trans B",
    "Naive B cell": "Naive B",
    "Memory B cell": "Memory B",
    "Plasma (Non-malignant)": "Plasma (Non-malignant)",
    "Plasma (Malignant)": "Plasma (Malignant)",
}

all_labels["aifi_plot_l2"] = (
    all_labels["aifi_label_l2"]
    .map(l2_to_plot2)
    .astype("category")
    .cat.remove_unused_categories()
)

## 7. Put everything together

In [16]:
all_labels = all_labels[all_labels["junk_cells"] == False]
all_labels = all_labels[
    [
        "aifi_celltype_l1",
        "aifi_label_l1",
        "aifi_celltype_l1.5",
        "aifi_label_l1.5",
        "aifi_celltype_l2",
        "aifi_label_l2",
        "aifi_plot_l2",
        "aifi_celltype_l3",
        "aifi_label_l3",
    ]
]

In [17]:
all_labels.to_parquet(
    '../../../data/rna/bmmc-labels/final-bmmc-labels.parquet')